In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
import time

In [48]:
# Define the state
class CrashState(TypedDict):
    input: str
    step1:str
    step2: str
    x: int



In [52]:
import time

# Define steps
def step_1(state : CrashState) -> CrashState :
  x = 2
  print("step1 executed")
  print("Initial Value of x in step_1 = ", x)
  return {"step1": "done", "input": state["input"], "x": x}

def step_2(state : CrashState) -> CrashState :
  print(" step 2 hanging ..... Now manually interrupt from Notebook")
  x = state.get("x", 0) + 1
  print("Value of the x in step_2 = ", x)
  time.sleep(30)   # To simulate long running hang
  return {"step2": "done", "x": x}

def step_3(state : CrashState) -> CrashState :
  print("step3 executed")
  x = state.get("x", 0) + 1
  print("Value of the x in step_3 = ", x)
  return {"step3": "done", "x": x}

In [53]:
# Build the LangGraph
builder = StateGraph(CrashState)
builder.add_node("step_1",step_1)
builder.add_node("step_2",step_2)
builder.add_node("step_3",step_3)

builder.set_entry_point("step_1")

builder.add_edge("step_1","step_2")
builder.add_edge("step_2","step_3")
builder.add_edge("step_3",END)

checkpointer= InMemorySaver()
graph=builder.compile(checkpointer=checkpointer)


In [54]:
try:
  print("....Running Graph......Pl. manually interrupt during step 2........")
  # We initialize "x" here in the input state dictionary
  graph.invoke({"input": "start", "x": 0}, config = {"configurable" : {"thread_id" : "thread-1"}})
except KeyboardInterrupt:
  print("Kernel manually interrupted............Crash simulated")

....Running Graph......Pl. manually interrupt during step 2........
step1 executed
Initial Value of x in step_1 =  2
 step 2 hanging ..... Now manually interrupt from Notebook
Value of the x in step_2 =  3
Kernel manually interrupted............Crash simulated


In [55]:
graph.get_state({"configurable" :{"thread_id" : "thread-1"}})

StateSnapshot(values={'input': 'start', 'step1': 'done', 'x': 2}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d053c-67fa-675f-8001-5bf1a14e1f6a'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-10-25T09:09:25.631343+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d053c-67f2-6efa-8000-a498f67df5f9'}}, tasks=(PregelTask(id='26b4b797-7369-3d3e-e949-e912609876b7', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [56]:
# Re-run to simulate Fault Tolerence
print("Running the graph again to demonstrate fault Tolerence")
final_state=graph.invoke(None,config = {"configurable" : {"thread_id" : "thread-1"}})  # Input as None as system restarted at crash point
print("\n Final State = ",final_state)              # The final step shows all steps in the workflow i.e. start -> input1 -> input2
                                                    # This proves workflow resumes from the crash point

Running the graph again to demonstrate fault Tolerence
 step 2 hanging ..... Now manually interrupt from Notebook
Value of the x in step_2 =  3
step3 executed
Value of the x in step_3 =  4

 Final State =  {'input': 'start', 'step1': 'done', 'step2': 'done', 'x': 4}


In [ ]:
graph.get_state({"configurable" :{"thread_id" : "thread-1"}})